# Практическая работа 4. Feature Engineering и регуляризация



## Подготовка данных
Загружаем California Housing, отделяем признаки от целевой переменной и делим данные на обучающую и тестовую выборки 80/20.


In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()

X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)


Обучающая выборка: (16512, 8)
Тестовая выборка: (4128, 8)


## Задание 1. Полиномиальные признаки
Создаём новые признаки степени 2 с помощью PolynomialFeatures.


In [2]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print("Количество исходных признаков:", X_train.shape[1])
print("Количество признаков после преобразования:", X_train_poly.shape[1])


Количество исходных признаков: 8
Количество признаков после преобразования: 44


include_bias=False ставим потому, что свободный коэффициент уже сама считает LinearRegression.

из 8 исходных признаков создаются:
- 8 исходных;
- квадраты каждого;
- попарные произведения признаков.

## Задание 2. Линейная регрессия на новых признаках
Сначала обучаем обычную линейную регрессию на исходных признаках, затем — на полиномиальных. Сравниваем R² на тестовой выборке.


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Модель на исходных признаках
model_base = LinearRegression()
model_base.fit(X_train, y_train)
y_pred_base = model_base.predict(X_test)
r2_base = r2_score(y_test, y_pred_base)

# Модель на полиномиальных признаках
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
y_pred_poly = model_poly.predict(X_test_poly)
r2_poly = r2_score(y_test, y_pred_poly)

print("R² обычной линейной регрессии:", r2_base)
print("R² линейной регрессии с полиномиальными признаками:", r2_poly)


R² обычной линейной регрессии: 0.5757877060324508
R² линейной регрессии с полиномиальными признаками: 0.6456819727519338


### Вывод по заданию 2
В задании 2 сравниваются две линейные регрессии: на исходных признаках и на полиномиальных. После добавления полиномиальных признаков R² вырос с 0.576 до 0.646, значит модель стала лучше описывать зависимость между признаками и стоимостью жилья. Но из-за увеличения числа признаков с 8 до 44 возрастает риск переобучения.


## Задание 3. Ridge Regression и подбор alpha
Применяем L2-регуляризацию Ridge и с помощью GridSearchCV подбираем лучшее значение alpha.



**Ridge-регуляризация**

Ridge помогает уменьшить риск переобучения.

Обычная линейная регрессия старается минимизировать ошибку модели.  
Ridge добавляет к этой ошибке штраф за слишком большие веса:

$$
Loss = MSE + \alpha \sum w_i^2
$$

где:

- $MSE$ — ошибка модели;
- $w_i$ — веса признаков;
- $\alpha$ — сила регуляризации.

Чем больше значение $\alpha$, тем сильнее модель уменьшает большие веса.

Это нужно потому, что при большом количестве признаков модель может слишком сильно подстроиться под обучающие данные.

Например, после PolynomialFeatures количество признаков увеличилось с 8 до 44.  
Из-за этого модель может начать учитывать случайные особенности обучающей выборки и давать некоторым признакам слишком большие веса.

В таком случае на обучающих данных ошибка будет маленькой, но на новых данных модель может работать хуже.

Это называется **переобучением**.

Ridge уменьшает веса и делает модель более устойчивой к новым данным.

In [4]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge = Ridge()

params = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(ridge, params, cv=5, scoring="r2")
grid.fit(X_train_poly, y_train)

best_ridge = grid.best_estimator_
y_pred_ridge = best_ridge.predict(X_test_poly)
r2_ridge = r2_score(y_test, y_pred_ridge)

print("Лучшее alpha:", grid.best_params_["alpha"])
print("R² Ridge:", r2_ridge)


/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.15557e-20): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.80539e-21): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.10753e-21): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.43414e-21): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.77785e-20): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

Лучшее alpha: 100
R² Ridge: 0.6417246825837529


/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.03064e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=6.35163e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.873e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


## Что такое alpha
alpha — сила регуляризации.

- маленькое alpha → модель почти как обычная линейная регрессия;
- большое alpha → веса сильнее уменьшаются;
- GridSearchCV проверяет несколько значений и выбирает лучшее.


## Задание 4. Сравнение весов
Сравниваем коэффициенты обычной линейной регрессии на полиномиальных признаках и Ridge.


In [9]:
weights = pd.DataFrame({
    "Признак": poly.get_feature_names_out(X.columns),
    "LinearRegression": model_poly.coef_,
    "Ridge": best_ridge.coef_
})

print(weights.head(15))


              Признак  LinearRegression     Ridge
0              MedInc        -11.936768 -0.187077
1            HouseAge         -0.842630 -0.378730
2            AveRooms          7.884154  0.016767
3           AveBedrms        -38.323120  0.010367
4          Population         -0.000228  0.000225
5            AveOccup          1.017431  0.029847
6            Latitude          8.284640  0.131315
7           Longitude          5.709792 -0.050909
8            MedInc^2         -0.030949 -0.024408
9     MedInc HouseAge          0.001577  0.003596
10    MedInc AveRooms          0.039547  0.031378
11   MedInc AveBedrms         -0.145653 -0.150688
12  MedInc Population          0.000051  0.000061
13    MedInc AveOccup         -0.004483 -0.006773
14    MedInc Latitude         -0.163992 -0.035904


### Вывод по заданию 4
После применения Ridge-регрессии большинство коэффициентов заметно уменьшилось по модулю. Это означает, что L2-регуляризация ограничивает слишком большие веса и делает модель менее чувствительной к отдельным признакам. Такой подход помогает снизить риск переобучения и делает модель более устойчивой.


## Дополнительное задание. L1-регуляризация Lasso
Обучаем Lasso и смотрим, какие коэффициенты стали равны нулю.


In [6]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=10000)
lasso.fit(X_train_poly, y_train)

y_pred_lasso = lasso.predict(X_test_poly)
r2_lasso = r2_score(y_test, y_pred_lasso)

zero_weights = (lasso.coef_ == 0).sum()

print("R² Lasso:", r2_lasso)
print("Количество обнулённых весов:", zero_weights)
print("Всего весов:", len(lasso.coef_))


R² Lasso: 0.6484033994232501
Количество обнулённых весов: 10
Всего весов: 44


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.721e+03, tolerance: 2.207e+00
  model = cd_fast.enet_coordinate_descent(


### Вывод по Lasso
Lasso использует L1-регуляризацию и может полностью обнулять некоторые коэффициенты.

То есть модель автоматически убирает часть признаков, которые считает менее полезными.


## Итоговое сравнение моделей


In [7]:
results = pd.DataFrame({
    "Модель": [
        "LinearRegression",
        "Polynomial LinearRegression",
        "Ridge",
        "Lasso"
    ],
    "R²": [
        r2_base,
        r2_poly,
        r2_ridge,
        r2_lasso
    ]
})

print(results)


                        Модель        R²
0             LinearRegression  0.575788
1  Polynomial LinearRegression  0.645682
2                        Ridge  0.641725
3                        Lasso  0.648403


### Общий вывод
В работе были созданы полиномиальные признаки степени 2 и обучена линейная регрессия на расширенном наборе признаков.

Затем была применена L2-регуляризация Ridge и подобрано значение alpha с помощью GridSearchCV. Ridge уменьшает веса модели и помогает бороться с переобучением.

Дополнительно была применена L1-регуляризация Lasso, которая может полностью обнулять некоторые коэффициенты.
